# C3 exp08 on v3: calibration and example order with few-shot prompts

Qwen2.5-7B-Instruct (4-bit), k = 0, definitions on, **data v3**: exp05's setting (1 solved train example per label; macro-F1 0.298 on v4). Zhao et al. (2021) find calibration helps most once examples are in the prompt, and that the **order** of the examples matters (the last example's label pulls the answer). exp05 always ends with `other`, and was never calibrated.

Four runs, each calibrated two ways afterwards (no re-scoring):
- **labels**: exp05's order (cs … other last), the control and the v3 baseline of this experiment (exp05 itself ran on v4);
- **reversed**: the same 7 examples, `cs` last;
- **shuffled**: the same 7 examples, order seed 1 (`bk, other, cc, aa, cs, ar, co`: `co` last);
- **shots2**: 2 examples per label (14), in label order.

Calibration targets: **uniform** (the paper, `_cc`) and the **train label frequencies** (`_ccp`: 84 % `other`, as in the data).

**Made faster** (exp05 alone took ~2 h):
- `--prefix-cache`: the instruction, options and examples are the same in every prompt, so they run once and only each segment's own tokens run per batch. Section 5 checks it on the 7B first (fp16 moves scores a little whenever batching changes; the cache must stay within 3 × the usual padded batch's own difference + 0.01); without it, the runs go without the cache, at batch 16.
- batch 32 with the cache (the cached batches are short), the two T4s in parallel, the model downloaded once, and one calibration process for all runs (one model load).

Note: `tasks/C3-prompting-baseline/experiments/exp08_calibration-order-7b/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached (only that one: the data cell looks for v3).

Expected time: about 1 hour with the cache, about 5 hours without it (estimates; the first progress lines of each log show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Download the model once
Both GPUs load the same weights: downloading them here first means neither run waits on the other's download.

In [ ]:
from huggingface_hub import snapshot_download

SEVEN_B = "Qwen/Qwen2.5-7B-Instruct"
print(snapshot_download(SEVEN_B))

## 5. Check the prefix cache on the 7B (~3 min)
16 val segments in the control run's setting: the 8 longest are scored with the cache, as a padded batch, and one at a time (the reference). The result decides the runs' flags below.

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/checks", ignore_errors=True)  # a throwaway check : always measured afresh
check = subprocess.run(
    ["python", "scripts/trial_prompting.py", "--data-root", DATA_ROOT, "--out-dir", "/kaggle/working/checks",
     "--run-name", "prefix_cache_check", "--model", SEVEN_B, "--load-in-4bit", "--prefix-cache",
     "--shots-per-class", "1", "--shot-seed", "0", "--limit", "16", "--batch-size", "16"],
    cwd="/kaggle/working/repo", env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src"),
    capture_output=True, text=True)
print("\n".join(l for l in check.stdout.splitlines() if l.startswith("prefix cache")))
if check.returncode != 0:
    print("\n".join(check.stderr.strip().splitlines()[-3:]))

USE_CACHE = check.returncode == 0
SPEED_FLAGS = ["--prefix-cache", "--batch-size", "32"] if USE_CACHE else ["--batch-size", "16"]
print("USE_CACHE =", USE_CACHE, SPEED_FLAGS, "" if USE_CACHE else ": the check failed , the runs go without the cache (~5 h)")

## 6. Helpers: a queue of runs on one GPU, and waiting while showing progress

In [ ]:
def queue(gpu, jobs):
    """run the jobs one after another on one GPU , in the background ; each run logs to logs/<run_name>.log"""
    commands = []
    for run_name, extra in jobs:
        commands.append(
            f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
            f"--run-name {run_name} --model {SEVEN_B} --load-in-4bit {' '.join(SPEED_FLAGS + extra)} "
            f"> /kaggle/working/logs/{run_name}.log 2>&1"
        )
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    # "; " : a failed run does not stop the next one in the queue
    return subprocess.Popen("; ".join(commands), shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:26} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 7. Run both queues
If the session stops, run this cell again: finished runs are skipped and an interrupted run continues where it stopped.
With the cache, each log starts with a `prefix cache :` line: each run repeats the check on its own longest prompts.

In [ ]:
ONE_SHOT = ["--shots-per-class", "1", "--shot-seed", "0"]
RUNS = ["c3_exp08_7b_labels", "c3_exp08_7b_reversed", "c3_exp08_7b_shuffled", "c3_exp08_7b_shots2"]
wait([
    queue(0, [
        ("c3_exp08_7b_labels", ONE_SHOT),
        ("c3_exp08_7b_shuffled", ONE_SHOT + ["--shot-order", "shuffled", "--shot-order-seed", "1"]),
    ]),
    queue(1, [
        ("c3_exp08_7b_reversed", ONE_SHOT + ["--shot-order", "reversed"]),
        ("c3_exp08_7b_shots2", ["--shots-per-class", "2", "--shot-seed", "0"]),
    ]),
])

## 8. Calibrate every finished run, both targets, with one model load
Runs already calibrated (after a restart) and runs that failed are skipped.

In [ ]:
finished = [f"/kaggle/working/runs/{r}" for r in RUNS if os.path.exists(f"/kaggle/working/runs/{r}/metrics.json")
            and not os.path.exists(f"/kaggle/working/runs/{r}_cc")]
print("to calibrate:", [os.path.basename(r) for r in finished])
if finished:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src")
    subprocess.run(["python", "scripts/calibrate_prompting.py", "--data-root", DATA_ROOT, "--run", *finished,
                    "--target", "uniform", "train-prior"], cwd="/kaggle/working/repo", env=env, check=True)

## 9. Results
Baseline: this notebook's `c3_exp08_7b_labels` (v3). For reference: exp05 0.298 (v4), exp03 7B zero-shot 0.206 (v3). Prompt noise (exp07): differences below ~0.035 are not real.

In [ ]:
def row(name):
    path = f"/kaggle/working/runs/{name}"
    if not os.path.exists(f"{path}/metrics.json"):
        return None
    m = json.load(open(f"{path}/metrics.json"))
    predicted_other = sum(json.loads(line)["label"] == "other" for line in open(f"{path}/predictions.jsonl"))
    return m["primary"]["macro_f1_6"], m["per_class"]["other"]["f1"], predicted_other, m

print(f"{'run':30} {'macro-F1':>8} {'other F1':>8} {'pred. other':>11}  (gold other 13,634)")
for name in RUNS:
    for suffix in ("", "_cc", "_ccp"):
        r = row(name + suffix)
        if r is None:
            print(f"{name + suffix:30} (missing : see logs/{name}.log)"); continue
        f1, other_f1, n_other, m = r
        print(f"{name + suffix:30} {f1:8.4f} {other_f1:8.3f} {n_other:11,}")
        print(" " * 30, {k: round(v["f1"], 3) for k, v in m["per_class"].items()})
    prefix = (row(name) or (0, 0, 0, {}))[3].get("prompting", {}).get("prefix_cache")
    print(" " * 30, "prefix cache:", prefix)

## 10. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp08_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp08_runs.zip runs logs
!ls -la c3_exp08_runs.zip